# Identificación de Clientes Mayoristas – Ahorrazo 2025

## ¿Qué hace este notebook?

Clasifica a cada cliente en uno de cuatro niveles según su comportamiento de compra:

| Nivel | Significado |
|---|---|
| **Alto** | Mayorista confirmado. Tickets grandes, frecuentes y alto gasto. |
| **Medio** | Comportamiento mayorista claro en al menos 2 de 3 dimensiones. |
| **Potencial** | Señal incipiente. Vale la pena hacer seguimiento. |
| **Minorista** | Sin señales mayoristas. |

## ¿Cómo se decide el nivel?

Se evalúan **3 criterios simples**, cada uno vale **1 punto**:

1. **UPT alto** – ¿El cliente lleva muchas unidades por visita?
2. **% tickets grandes alto** – ¿La mayoría de sus compras son de gran volumen?
3. **Gasto total alto** – ¿Es un cliente de alto valor monetario?

El corte de cada criterio se calcula automáticamente usando el **Q3 (percentil 75)** del universo de clientes.  
Eso significa que los umbrales se adaptan solos a los datos reales de cada período.

**Puntaje → Nivel:**
- 3 puntos → **Alto**
- 2 puntos → **Medio**
- 1 punto  → **Potencial**
- 0 puntos → **Minorista**

---
> **Nota sobre sucursales:** el análisis se corre tanto de forma global (todos los clientes en todas las sucursales)  
> como por sucursal individual (SL, R1, R2). Los umbrales se recalculan en cada contexto.

## 0. Librerías

In [14]:
import time
import pyodbc
import pandas as pd
import numpy as np

from urllib.parse import quote_plus
from sqlalchemy import create_engine
from sqlalchemy.engine import Engine

## 1. Configuración general

Todo lo que se puede ajustar está acá arriba, en un solo lugar.

In [3]:
# ── Período de análisis ──────────────────────────────────────────────────────
FECHA_INICIO = "2025-01-01"
FECHA_FIN    = "2026-01-01"   # exclusivo (< esta fecha)

# ── Umbral para % tickets grandes ───────────────────────────────────────────
# El cliente debe tener AL MENOS esta fracción de tickets grandes para cumplir C2.
# El corte de qué es un 'ticket grande' se calcula automáticamente con
# el método del boxplot: Q3 + 1.5 × IQR sobre la distribución de unidades
# por ticket de TODO el universo. No es un número fijo: se adapta a los datos.
PCT_GRANDES_MINIMO = 0.40   # 30 %

# ── Clientes y productos a excluir ──────────────────────────────────────────
# Cliente genérico / sin identificar
EXCLUIR_CLIENTES = ["44444401-7"]

# Bolsas, servicios y productos sin valor analítico
EXCLUIR_PRODUCTOS = [
    "442", "999994", "31852", "999992", "90311",
    "999998", "901762", "9809999", "51457", "77190",
    "8925", "91988", "999995"
]

# ── Mapa de sucursales (pdv_id → nombre legible) ────────────────────────────
NOMBRE_SUCURSAL = {
    "3": "Ahorrazo SL",
    "5": "Ahorrazo R1",
    "6": "Ahorrazo R2",
}

## 2. Conexión y carga de datos

In [4]:
def conexion_bd():
    """Abre la conexión a SQL Server. Detecta automáticamente el driver disponible."""
    drivers = pyodbc.drivers()
    preferidos = ["ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server"]
    driver = next((d for d in preferidos if d in drivers), None)
    if not driver:
        raise RuntimeError(f"No se encontró driver ODBC SQL Server. Detectados: {drivers}")
    conn_str = (
        f"DRIVER={{{driver}}};"
        f"SERVER=192.168.5.231;"
        f"DATABASE=DATO_SOLUTIONS;"
        f"UID=sa;"
        f"PWD=Server_123456;"
        f"Encrypt=yes;TrustServerCertificate=yes;"
    )
    return pyodbc.connect(conn_str)


SQL = """
SELECT * FROM dbo.view_ventas_ahorrazo_filtradas_12m;
"""

with conexion_bd() as conn:
    ventas_raw = pd.read_sql_query(SQL, conn, params=[FECHA_INICIO, FECHA_FIN])

print(f"Filas cargadas: {len(ventas_raw):,}")
ventas_raw.head()

C:\Users\Administrador\AppData\Local\Temp\2\ipykernel_1708\4058972122.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ventas_raw = pd.read_sql_query(SQL, conn, params=[FECHA_INICIO, FECHA_FIN])


Filas cargadas: 69,853,749


,fecha_venta,cliente_id,producto_id,pdv_id,ticket_id,unidades,venta_gs,cliente_id_limpio,nombre_cliente
0,2025-01-02,6996666-4,1001051,3,15673870_003-031-0147361,1.0,19450.0,6996666,BAEZ CUENCA ARCELIA
1,2025-01-02,998619-7,1001080,6,15673870_006-022-0018301,1.0,11600.0,998619,VICTOR VELAZQUEZ
2,2025-01-02,2879504-0,1001090,6,15673870_006-004-0020072,1.0,22250.0,2879504,MIRIAN RUIZ
3,2025-01-02,3697732-2,1001090,3,15673870_003-026-0169323,1.0,22250.0,3697732,RINCON TARJETA
4,2025-01-02,3697732-2,1001090,3,15673870_003-026-0169324,1.0,22250.0,3697732,RINCON TARJETA


## 3. Limpieza

Se excluyen el cliente genérico y productos sin valor analítico (bolsas, servicios, etc.).

In [5]:
ventas = ventas_raw.copy()

# Tipos
ventas["fecha_venta"]       = pd.to_datetime(ventas["fecha_venta"])
ventas["cliente_id"]        = ventas["cliente_id"].astype(str)
ventas["cliente_id_limpio"] = ventas["cliente_id_limpio"].astype(str)
ventas["producto_id"]       = ventas["producto_id"].astype(str)
ventas["pdv_id"]            = ventas["pdv_id"].astype(str)

# Filtros de exclusión
ventas = ventas[~ventas["cliente_id"].isin(EXCLUIR_CLIENTES)]
ventas = ventas[~ventas["producto_id"].isin(EXCLUIR_PRODUCTOS)]

print(f"Filas después de limpieza: {len(ventas):,}")
print(f"Clientes únicos:           {ventas['cliente_id_limpio'].nunique():,}")

Filas después de limpieza: 64,879,590
Clientes únicos:           421,233


## 4. Funciones principales

### 4.1. Calcular métricas por cliente

Para cada cliente (y opcionalmente por sucursal) se calculan **3 métricas base**:

- **UPT** (Unidades por Ticket): promedio de unidades que lleva en cada visita.
- **% Tickets Grandes**: proporción de tickets donde llevó más de el umbral boxplot (Q3 + 1.5×IQR del universo).
- **Ventas Totales**: gasto acumulado en el período.

In [6]:
def calcular_umbral_ticket_grande(df: pd.DataFrame) -> float:
    """
    Calcula el umbral de 'ticket grande' usando el método del boxplot:
        umbral = Q3 + 1.5 × IQR

    Se aplica sobre la distribución de unidades por ticket de TODO el universo
    (no por cliente), excluyendo tickets con unidades ≤ 0 (devoluciones).

    ¿Por qué este método?
    El boxplot define 'valor atípicamente alto' como todo lo que supera
    Q3 + 1.5×IQR. Eso significa que el umbral se adapta automáticamente
    a la distribución real: si hay clientes con 1000 unidades y otros con 5,
    el IQR captura la dispersión y el umbral queda en un punto naturalmente
    elevado, sin que los extremos lo distorsionen.
    """
    upt_todos = (
        df.groupby(["ticket_id"], dropna=False)["unidades"]
          .sum()
    )
    upt_todos = upt_todos[upt_todos > 0]  # excluir devoluciones

    q1 = upt_todos.quantile(0.25)
    q3 = upt_todos.quantile(0.75)
    iqr = q3 - q1
    umbral = q3 + 1.5 * iqr

    print(f"  Distribución de unidades/ticket → "
          f"Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}")
    print(f"  Umbral ticket grande (Q3 + 1.5×IQR) = {umbral:.1f} unidades")

    return float(umbral)


def calcular_metricas(df: pd.DataFrame, group_cols: list,
                      umbral_ticket_grande: float) -> pd.DataFrame:
    """
    Calcula las 3 métricas base por cliente (o cliente × sucursal).

    Parámetros
    ----------
    df                   : DataFrame de ventas ya limpio.
    group_cols           : Columnas de agrupación.
                           ['cliente_id_limpio']            → análisis global
                           ['pdv_id', 'cliente_id_limpio']  → análisis por sucursal
    umbral_ticket_grande : Número de unidades a partir del cual un ticket
                           se considera 'grande'. Calculado con Q3 + 1.5×IQR.

    Retorna
    -------
    DataFrame con una fila por cliente (o cliente × sucursal) y las columnas:
      - tickets_unicos, dias_con_compra
      - upt                : unidades promedio por ticket
      - pct_tickets_grandes: fracción de tickets que superan umbral_ticket_grande
      - ventas_totales     : suma de venta_gs
    """
    # ── Paso 1: unidades por ticket (nivel ticket, no línea de detalle) ──────
    # Primero sumamos unidades del cliente dentro de cada ticket para obtener
    # el tamaño real de cada compra. Excluimos tickets con unidades ≤ 0
    # (devoluciones) para no distorsionar el análisis de comportamiento.
    upt_por_ticket = (
        df.groupby(group_cols + ["ticket_id"], dropna=False)["unidades"]
          .sum()
          .reset_index()
          .rename(columns={"unidades": "upt"})
    )
    upt_por_ticket = upt_por_ticket[upt_por_ticket["upt"] > 0]

    # ── Paso 2: métricas de comportamiento por cliente ───────────────────────
    metricas_ticket = (
        upt_por_ticket
          .groupby(group_cols, dropna=False)
          .agg(
              tickets_unicos  = ("ticket_id", "count"),
              upt_suma        = ("upt",       "sum"),
              # Un ticket es 'grande' si supera el umbral boxplot del universo
              tickets_grandes = ("upt", lambda x: (x > umbral_ticket_grande).sum()),
          )
          .reset_index()
    )

    # UPT promedio = total unidades / total tickets
    metricas_ticket["upt"] = (
        metricas_ticket["upt_suma"] / metricas_ticket["tickets_unicos"]
    ).fillna(0.0)

    # % tickets grandes = tickets que superan el umbral boxplot / total tickets
    metricas_ticket["pct_tickets_grandes"] = (
        metricas_ticket["tickets_grandes"] / metricas_ticket["tickets_unicos"]
    ).fillna(0.0)

    # ── Paso 3: días con compra y ventas totales (del df original) ───────────
    otras = (
        df.groupby(group_cols, dropna=False)
          .agg(
              dias_con_compra = ("fecha_venta", "nunique"),
              ventas_totales  = ("venta_gs",    "sum"),
          )
          .reset_index()
    )

    # ── Paso 4: unir y devolver columnas útiles ──────────────────────────────
    out = metricas_ticket.merge(otras, on=group_cols, how="left")
    cols_finales = group_cols + [
        "tickets_unicos", "dias_con_compra",
        "upt", "pct_tickets_grandes", "ventas_totales"
    ]
    return out[cols_finales]

### 4.2. Clasificar por nivel mayorista

Con las métricas calculadas, se evalúan los 3 criterios y se asigna el nivel.

Los umbrales son **dinámicos**: se calculan como el Q3 del universo analizado,  
con un piso mínimo para evitar umbrales demasiado bajos en universos pequeños.

In [7]:
def clasificar_nivel(metricas: pd.DataFrame) -> pd.DataFrame:
    """
    Asigna un puntaje (0–3) y un nivel mayorista a cada cliente.

    Criterios (1 punto cada uno):
      C1 – UPT alto           : upt              ≥ Q3_upt  (del universo de clientes)
      C2 – % grandes alto     : pct_tick_grandes ≥ max(PCT_GRANDES_MINIMO, Q3_pct)
      C3 – Ventas totales alto : ventas_totales   ≥ Q3_ventas

    Niveles:
      3 → Alto      2 → Medio      1 → Potencial      0 → Minorista
    """
    out = metricas.copy()

    # ── Umbrales dinámicos (Q3 del universo actual) ──────────────────────────
    q3_upt     = out["upt"].quantile(0.75)
    q3_pct     = out["pct_tickets_grandes"].quantile(0.75)
    q3_ventas  = out["ventas_totales"].quantile(0.75)

    umbral_upt    = q3_upt
    umbral_pct    = max(PCT_GRANDES_MINIMO,  q3_pct)
    umbral_ventas = q3_ventas

    # ── Criterios binarios (True = 1 punto, False = 0) ───────────────────────
    out["c1_upt_alto"]         = out["upt"]                >= umbral_upt
    out["c2_pct_grandes_alto"] = out["pct_tickets_grandes"] >= umbral_pct
    out["c3_ventas_alto"]      = out["ventas_totales"]      >= umbral_ventas

    # ── Puntaje total (0–3) ──────────────────────────────────────────────────
    out["puntaje"] = (
        out["c1_upt_alto"].astype(int)
        + out["c2_pct_grandes_alto"].astype(int)
        + out["c3_ventas_alto"].astype(int)
    )

    # ── Nivel mayorista ──────────────────────────────────────────────────────
    def _nivel(p):
        if p == 3: return "Alto"
        if p == 2: return "Medio"
        if p == 1: return "Potencial"
        return "Minorista"

    out["nivel_mayorista"] = out["puntaje"].apply(_nivel)

    # Orden categórico para que sort funcione bien en reportes
    out["nivel_mayorista"] = pd.Categorical(
        out["nivel_mayorista"],
        categories=["Alto", "Medio", "Potencial", "Minorista"],
        ordered=True
    )

    # ── Guardar los umbrales usados (útil para auditoría y reportes) ─────────
    out["umbral_upt_usado"]    = round(umbral_upt,    2)
    out["umbral_pct_usado"]    = round(umbral_pct,    4)
    out["umbral_ventas_usado"] = round(umbral_ventas, 0)

    return out

### 4.3. Pipeline completo: global + por sucursal

In [8]:
def correr_analisis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ejecuta el análisis completo: global y por cada sucursal.
    Devuelve un único DataFrame con columna 'scope' que indica el contexto.

    El umbral de 'ticket grande' se calcula UNA SOLA VEZ sobre todo el universo
    de tickets (Q3 + 1.5×IQR). Ese mismo umbral se usa en todos los scopes
    para que los resultados sean comparables entre sucursales.
    """
    resultados = []

    # ── Calcular umbral de ticket grande sobre el universo completo ──────────
    print("Calculando umbral de ticket grande sobre el universo completo:")
    umbral_tg = calcular_umbral_ticket_grande(df)
    print()

    # ── Análisis global (todas las sucursales juntas) ────────────────────────
    print("Procesando: GLOBAL")
    metricas_global = calcular_metricas(df, group_cols=["cliente_id_limpio"],
                                        umbral_ticket_grande=umbral_tg)
    clasificado_global = clasificar_nivel(metricas_global)
    clasificado_global.insert(0, "scope", "GLOBAL")
    clasificado_global["umbral_ticket_grande"] = round(umbral_tg, 1)
    resultados.append(clasificado_global)

    # ── Análisis por sucursal ────────────────────────────────────────────────
    for pdv_id, nombre in NOMBRE_SUCURSAL.items():
        df_suc = df[df["pdv_id"] == pdv_id]
        if df_suc.empty:
            print(f"\u26a0\ufe0f  Sin datos para sucursal {nombre} (pdv_id={pdv_id})")
            continue
        print(f"Procesando: {nombre}")
        metricas_suc = calcular_metricas(df_suc, group_cols=["cliente_id_limpio"],
                                         umbral_ticket_grande=umbral_tg)
        clasificado_suc = clasificar_nivel(metricas_suc)
        clasificado_suc.insert(0, "scope", nombre)
        clasificado_suc["umbral_ticket_grande"] = round(umbral_tg, 1)
        resultados.append(clasificado_suc)

    # ── Unir todo ────────────────────────────────────────────────────────────
    out = pd.concat(resultados, ignore_index=True)

    # Agregar nombre del cliente (si existe en los datos originales)
    nombres = (
        df[["cliente_id_limpio", "nombre_cliente"]]
          .drop_duplicates(subset="cliente_id_limpio")
    ) if "nombre_cliente" in df.columns else None

    if nombres is not None:
        out = out.merge(nombres, on="cliente_id_limpio", how="left")

    # Ordenar columnas de forma legible
    cols_primeras = [
        "scope", "cliente_id_limpio", "nombre_cliente",
        "nivel_mayorista", "puntaje",
        "c1_upt_alto", "c2_pct_grandes_alto", "c3_ventas_alto",
        "upt", "pct_tickets_grandes", "ventas_totales",
        "tickets_unicos", "dias_con_compra",
        "umbral_ticket_grande",
        "umbral_upt_usado", "umbral_pct_usado", "umbral_ventas_usado",
    ]
    cols_primeras = [c for c in cols_primeras if c in out.columns]
    out = out[cols_primeras]

    return out.sort_values(["scope", "nivel_mayorista", "ventas_totales"],
                           ascending=[True, True, False]).reset_index(drop=True)


## 5. Ejecutar el análisis

In [9]:
resultado = correr_analisis(ventas)

print(f"Total de registros (clientes × scope): {len(resultado):,}")
resultado.head(10)

Calculando umbral de ticket grande sobre el universo completo:
  Distribución de unidades/ticket → Q1=4.0  Q3=24.1  IQR=20.1
  Umbral ticket grande (Q3 + 1.5×IQR) = 54.2 unidades

Procesando: GLOBAL
Procesando: Ahorrazo SL
Procesando: Ahorrazo R1
Procesando: Ahorrazo R2
Total de registros (clientes × scope): 951,923


,scope,cliente_id_limpio,nombre_cliente,nivel_mayorista,puntaje,c1_upt_alto,c2_pct_grandes_alto,c3_ventas_alto,upt,pct_tickets_grandes,ventas_totales,tickets_unicos,dias_con_compra,umbral_ticket_grande,umbral_upt_usado,umbral_pct_usado,umbral_ventas_usado
0,Ahorrazo R1,3699201,ZUNILDA GAUTO DE AGUERO,Alto,3,True,True,True,56.521582,0.543434,4.214400e+08,495,352,54.2,25.0,0.4,1353352.0
1,Ahorrazo R1,4132638,MARIA LETICIA LOPEZ MARIN,Alto,3,True,True,True,454.875486,0.861111,4.130557e+08,144,120,54.2,25.0,0.4,1353352.0
2,Ahorrazo R1,2444512,DAVID GUSTAVO AYALA LARA,Alto,3,True,True,True,66.618048,0.527881,3.544497e+08,269,220,54.2,25.0,0.4,1353352.0
3,Ahorrazo R1,6348789,VEGA SALDIVAR CATALINO,Alto,3,True,True,True,143.434980,0.813869,3.191456e+08,274,228,54.2,25.0,0.4,1353352.0
4,Ahorrazo R1,3038856,ARMINDA BOGADO ORTEGA,Alto,3,True,True,True,72.219773,0.712895,2.184145e+08,411,333,54.2,25.0,0.4,1353352.0
5,Ahorrazo R1,4018867,PERLA NOEMI DASPETT LARA,Alto,3,True,True,True,128.821268,0.747368,1.630368e+08,190,154,54.2,25.0,0.4,1353352.0
6,Ahorrazo R1,1964804,VICTOR ANTONIO DELGADO NOGUERA,Alto,3,True,True,True,497.902439,0.951220,1.382333e+08,41,38,54.2,25.0,0.4,1353352.0
7,Ahorrazo R1,3414986,BLASIDA SALINAS,Alto,3,True,True,True,48.623980,0.421512,1.263752e+08,344,280,54.2,25.0,0.4,1353352.0
8,Ahorrazo R1,3652224,ROMERO EDWARDS HUGO ANDRES,Alto,3,True,True,True,122.709688,1.000000,1.226446e+08,16,16,54.2,25.0,0.4,1353352.0
9,Ahorrazo R1,2523744,LIZ CANO,Alto,3,True,True,True,334.165800,0.860000,1.146477e+08,50,44,54.2,25.0,0.4,1353352.0


## 6. Resumen ejecutivo

Distribución de niveles por scope (cuántos clientes hay en cada nivel).

In [10]:
resumen = (
    resultado
      .groupby(["scope", "nivel_mayorista"], observed=True)
      .agg(
          clientes     = ("cliente_id_limpio", "nunique"),
          ventas_gs    = ("ventas_totales",    "sum"),
      )
      .reset_index()
)

# % de clientes por nivel dentro de cada scope
resumen["pct_clientes"] = (
    resumen.groupby("scope")["clientes"]
           .transform(lambda x: x / x.sum() * 100)
           .round(1)
)

resumen["ventas_gs"] = resumen["ventas_gs"].map("{:,.0f}".format)

print(resumen.to_string(index=False))

      scope nivel_mayorista  clientes       ventas_gs  pct_clientes
Ahorrazo R1            Alto      6078  48,926,501,723           3.6
Ahorrazo R1           Medio     20681  98,365,715,362          12.1
Ahorrazo R1       Potencial     38995 122,369,908,529          22.8
Ahorrazo R1       Minorista    105227  29,926,229,816          61.5
Ahorrazo R2            Alto      4785  32,246,930,622           3.3
Ahorrazo R2           Medio     17564  82,455,637,144          12.0
Ahorrazo R2       Potencial     34759 107,070,973,797          23.7
Ahorrazo R2       Minorista     89626  25,979,929,909          61.1
Ahorrazo SL            Alto      7126  35,940,857,560           3.3
Ahorrazo SL           Medio     24640  91,614,542,932          11.6
Ahorrazo SL       Potencial     48311  90,066,806,732          22.7
Ahorrazo SL       Minorista    132939  25,408,763,731          62.4
     GLOBAL            Alto     12175 107,361,586,129           2.9
     GLOBAL           Medio     51880 285,744,73

## 7. Detalle por scope

Lista de clientes mayoristaas (Alto o Medio) para cada contexto.

In [11]:
def ver_mayoristas(scope: str, niveles=("Alto", "Medio")):
    """
    Muestra los clientes mayoristas de un scope dado.

    Ejemplos de uso:
        ver_mayoristas("GLOBAL")
        ver_mayoristas("Ahorrazo SL", niveles=("Alto",))
        ver_mayoristas("Ahorrazo R1", niveles=("Alto", "Medio", "Potencial"))
    """
    df = resultado[
        (resultado["scope"] == scope) &
        (resultado["nivel_mayorista"].isin(niveles))
    ].copy()

    print(f"\n{'─'*60}")
    print(f"  {scope} – {', '.join(niveles)} ({len(df)} clientes)")
    print(f"{'─'*60}")
    display(df.reset_index(drop=True))


# Ejecutar para cada scope
ver_mayoristas("GLOBAL")
ver_mayoristas("Ahorrazo SL")
ver_mayoristas("Ahorrazo R1")
ver_mayoristas("Ahorrazo R2")


────────────────────────────────────────────────────────────
  GLOBAL – Alto, Medio (64055 clientes)
────────────────────────────────────────────────────────────


,scope,cliente_id_limpio,nombre_cliente,nivel_mayorista,puntaje,c1_upt_alto,c2_pct_grandes_alto,c3_ventas_alto,upt,pct_tickets_grandes,ventas_totales,tickets_unicos,dias_con_compra,umbral_ticket_grande,umbral_upt_usado,umbral_pct_usado,umbral_ventas_usado
0,GLOBAL,2432217,ROLANDO MANUEL PACHECO BARRIOS,Alto,3,True,True,True,507.863444,0.821192,4.696288e+08,151,84,54.2,23.32,0.4,1534008.0
1,GLOBAL,4132638,MARIA LETICIA LOPEZ MARIN,Alto,3,True,True,True,444.946419,0.858108,4.263667e+08,148,121,54.2,23.32,0.4,1534008.0
2,GLOBAL,3699201,ZUNILDA GAUTO DE AGUERO,Alto,3,True,True,True,56.434270,0.542339,4.218147e+08,496,352,54.2,23.32,0.4,1534008.0
3,GLOBAL,3235169,SILVA DE LIN GLADYS RAMONA,Alto,3,True,True,True,112.107348,0.811236,3.870671e+08,445,358,54.2,23.32,0.4,1534008.0
4,GLOBAL,2444512,DAVID GUSTAVO AYALA LARA,Alto,3,True,True,True,66.285565,0.521595,3.843279e+08,301,238,54.2,23.32,0.4,1534008.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64050,GLOBAL,5727436,NISA SHAIKH,Medio,2,True,True,False,80.000000,1.000000,6.000000e+04,1,1,54.2,23.32,0.4,1534008.0
64051,GLOBAL,5019950,GARCETE DUARTE DAHIANA SOLEDAD,Medio,2,True,True,False,75.000000,1.000000,5.565000e+04,1,1,54.2,23.32,0.4,1534008.0
64052,GLOBAL,1113533,ZARATE AQUINO MARIA FELICITA,Medio,2,True,True,False,120.000000,1.000000,3.600000e+04,1,1,54.2,23.32,0.4,1534008.0
64053,GLOBAL,80010933,IMAGEN MEDICA SRL,Medio,2,True,True,False,71.000000,1.000000,2.130000e+04,1,1,54.2,23.32,0.4,1534008.0



────────────────────────────────────────────────────────────
  Ahorrazo SL – Alto, Medio (31766 clientes)
────────────────────────────────────────────────────────────


,scope,cliente_id_limpio,nombre_cliente,nivel_mayorista,puntaje,c1_upt_alto,c2_pct_grandes_alto,c3_ventas_alto,upt,pct_tickets_grandes,ventas_totales,tickets_unicos,dias_con_compra,umbral_ticket_grande,umbral_upt_usado,umbral_pct_usado,umbral_ventas_usado
0,Ahorrazo SL,3235169,SILVA DE LIN GLADYS RAMONA,Alto,3,True,True,True,113.471981,0.822323,3.862164e+08,439,358,54.2,20.79,0.4,817045.0
1,Ahorrazo SL,4320665,HILDA SOSA,Alto,3,True,True,True,70.401376,0.690037,1.633023e+08,271,256,54.2,20.79,0.4,817045.0
2,Ahorrazo SL,430213,PEDRO GARCETE,Alto,3,True,True,True,309.679776,0.931034,1.487651e+08,58,56,54.2,20.79,0.4,817045.0
3,Ahorrazo SL,80013551,METALURGICA VERA SRL,Alto,3,True,True,True,1527.033333,0.800000,1.481440e+08,15,13,54.2,20.79,0.4,817045.0
4,Ahorrazo SL,943095,RUBEN ACOSTA,Alto,3,True,True,True,54.312041,0.441296,1.457624e+08,247,241,54.2,20.79,0.4,817045.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31761,Ahorrazo SL,4592792,NORMA MORA,Medio,2,True,True,False,60.000000,1.000000,6.000000e+04,1,1,54.2,20.79,0.4,817045.0
31762,Ahorrazo SL,5727436,NISA SHAIKH,Medio,2,True,True,False,80.000000,1.000000,6.000000e+04,1,1,54.2,20.79,0.4,817045.0
31763,Ahorrazo SL,1113533,ZARATE AQUINO MARIA FELICITA,Medio,2,True,True,False,120.000000,1.000000,3.600000e+04,1,1,54.2,20.79,0.4,817045.0
31764,Ahorrazo SL,80010933,IMAGEN MEDICA SRL,Medio,2,True,True,False,71.000000,1.000000,2.130000e+04,1,1,54.2,20.79,0.4,817045.0



────────────────────────────────────────────────────────────
  Ahorrazo R1 – Alto, Medio (26759 clientes)
────────────────────────────────────────────────────────────


,scope,cliente_id_limpio,nombre_cliente,nivel_mayorista,puntaje,c1_upt_alto,c2_pct_grandes_alto,c3_ventas_alto,upt,pct_tickets_grandes,ventas_totales,tickets_unicos,dias_con_compra,umbral_ticket_grande,umbral_upt_usado,umbral_pct_usado,umbral_ventas_usado
0,Ahorrazo R1,3699201,ZUNILDA GAUTO DE AGUERO,Alto,3,True,True,True,56.521582,0.543434,4.214400e+08,495,352,54.2,25.0,0.4,1353352.0
1,Ahorrazo R1,4132638,MARIA LETICIA LOPEZ MARIN,Alto,3,True,True,True,454.875486,0.861111,4.130557e+08,144,120,54.2,25.0,0.4,1353352.0
2,Ahorrazo R1,2444512,DAVID GUSTAVO AYALA LARA,Alto,3,True,True,True,66.618048,0.527881,3.544497e+08,269,220,54.2,25.0,0.4,1353352.0
3,Ahorrazo R1,6348789,VEGA SALDIVAR CATALINO,Alto,3,True,True,True,143.434980,0.813869,3.191456e+08,274,228,54.2,25.0,0.4,1353352.0
4,Ahorrazo R1,3038856,ARMINDA BOGADO ORTEGA,Alto,3,True,True,True,72.219773,0.712895,2.184145e+08,411,333,54.2,25.0,0.4,1353352.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26754,Ahorrazo R1,3006051,FRANCISCO LIBRADO GONZALEZ,Medio,2,True,True,False,87.000000,1.000000,7.045000e+04,1,1,54.2,25.0,0.4,1353352.0
26755,Ahorrazo R1,6011428,LARIZZA BAREIRO,Medio,2,True,True,False,102.000000,1.000000,6.655000e+04,1,1,54.2,25.0,0.4,1353352.0
26756,Ahorrazo R1,4648700,YOHANA BELEN MERELES VERA,Medio,2,True,True,False,97.000000,1.000000,6.565000e+04,1,1,54.2,25.0,0.4,1353352.0
26757,Ahorrazo R1,1676964,ROMERO DE VERGARA ANGELINA,Medio,2,True,True,False,64.000000,1.000000,6.545000e+04,1,1,54.2,25.0,0.4,1353352.0



────────────────────────────────────────────────────────────
  Ahorrazo R2 – Alto, Medio (22349 clientes)
────────────────────────────────────────────────────────────


,scope,cliente_id_limpio,nombre_cliente,nivel_mayorista,puntaje,c1_upt_alto,c2_pct_grandes_alto,c3_ventas_alto,upt,pct_tickets_grandes,ventas_totales,tickets_unicos,dias_con_compra,umbral_ticket_grande,umbral_upt_usado,umbral_pct_usado,umbral_ventas_usado
0,Ahorrazo R2,2432217,ROLANDO MANUEL PACHECO BARRIOS,Alto,3,True,True,True,512.001163,0.813953,3.991122e+08,129,65,54.2,25.18,0.4,1359459.0
1,Ahorrazo R2,1918730,SALDIVAR PAREDEZ NORMA BEATRIZ,Alto,3,True,True,True,155.696977,0.550926,1.915435e+08,216,111,54.2,25.18,0.4,1359459.0
2,Ahorrazo R2,3778362,VILLAGRA HERMOSILLA GRACIELA RAMONA,Alto,3,True,True,True,324.969430,0.860759,1.823645e+08,79,60,54.2,25.18,0.4,1359459.0
3,Ahorrazo R2,945157,LOPEZ DE VILLALBA DALMIHT ENCARNACION,Alto,3,True,True,True,74.774981,0.404580,1.134001e+08,262,147,54.2,25.18,0.4,1359459.0
4,Ahorrazo R2,3852216,MARIA RAQUEL AGUERO LOPEZ,Alto,3,True,True,True,139.788551,0.666667,9.160760e+07,138,112,54.2,25.18,0.4,1359459.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22344,Ahorrazo R2,4846453,VICTORIA NOEMI VALENZUELA AMARILLA,Medio,2,True,True,False,55.000000,1.000000,6.825000e+04,1,1,54.2,25.18,0.4,1359459.0
22345,Ahorrazo R2,1548059,ELVA AGUSTINA RAMOS LARREA,Medio,2,True,True,False,75.000000,1.000000,6.700000e+04,1,1,54.2,25.18,0.4,1359459.0
22346,Ahorrazo R2,2057905,VIRGINIA SALINAS,Medio,2,True,True,False,66.000000,1.000000,6.685000e+04,1,1,54.2,25.18,0.4,1359459.0
22347,Ahorrazo R2,2037825,ALFREDO FUNES,Medio,2,True,True,False,99.000000,1.000000,6.195000e+04,1,1,54.2,25.18,0.4,1359459.0


## 8. Verificación: ¿qué umbrales se usaron?

Permite auditar exactamente qué corte se aplicó en cada scope.

In [12]:
umbrales_usados = (
    resultado
      .groupby("scope")[["umbral_upt_usado", "umbral_pct_usado", "umbral_ventas_usado"]]
      .first()
      .reset_index()
)

umbrales_usados.columns = ["Scope", "Umbral UPT", "Umbral % Grandes", "Umbral Ventas (Gs.)"]
umbrales_usados["Umbral Ventas (Gs.)"] = umbrales_usados["Umbral Ventas (Gs.)"].map("{:,.0f}".format)

print("Umbrales dinámicos aplicados (Q3 de cada universo):")
print(umbrales_usados.to_string(index=False))

Umbrales dinámicos aplicados (Q3 de cada universo):
      Scope  Umbral UPT  Umbral % Grandes Umbral Ventas (Gs.)
Ahorrazo R1       25.00               0.4           1,353,352
Ahorrazo R2       25.18               0.4           1,359,459
Ahorrazo SL       20.79               0.4             817,045
     GLOBAL       23.32               0.4           1,534,008


## 9. Exportar resultados (opcional)

In [ ]:
# Descomentar para exportar a Excel
# resultado.to_excel("mayoristas_ahorrazo_2025.xlsx", index=False)
# print("Archivo exportado: mayoristas_ahorrazo_2025.xlsx")

In [17]:
def engine_sqlserver():
    drivers = pyodbc.drivers()
    preferidos = ["ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server"]
    driver = next((d for d in preferidos if d in drivers), None)
    if not driver:
        raise RuntimeError(f"No se encontró driver ODBC SQL Server. Detectados: {drivers}")

    odbc_str = (
        f"DRIVER={{{driver}}};"
        f"SERVER=192.168.5.231;"
        f"DATABASE=DATO_SOLUTIONS;"
        f"UID=sa;"
        f"PWD=Server_123456;"
        f"Encrypt=yes;"
        f"TrustServerCertificate=yes;"
    )

    params = quote_plus(odbc_str)
    return create_engine(
        f"mssql+pyodbc:///?odbc_connect={params}",
        fast_executemany=True
    )

table_name = 'clientes_mayoristas_v2'
engine = engine_sqlserver()
try:
    resultado.to_sql(
        table_name,
        schema="dbo",
        con=engine,
        if_exists='replace',
        index=False,
        chunksize=50000
    )
    print(f"Dataframe successfully written to table {table_name}")
except Exception as e:
    print(f"An error ocurred: {e}")

An error ocurred: (pyodbc.Error) ('HY000', '[HY000] [Microsoft][ODBC Driver 17 for SQL Server]Unspecified error occurred on SQL Server. Connection may have been terminated by the server. (0) (SQLExecDirectW); [HY000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot continue the execution because the session is in the kill state. (596)')
[SQL: SELECT [INFORMATION_SCHEMA].[TABLES].[TABLE_NAME] 
FROM [INFORMATION_SCHEMA].[TABLES] 
WHERE [INFORMATION_SCHEMA].[TABLES].[TABLE_SCHEMA] = CAST(? AS NVARCHAR(max)) AND [INFORMATION_SCHEMA].[TABLES].[TABLE_TYPE] = CAST(? AS NVARCHAR(max)) ORDER BY [INFORMATION_SCHEMA].[TABLES].[TABLE_NAME]]
[parameters: ('dbo', 'BASE TABLE')]
(Background on this error at: https://sqlalche.me/e/20/dbapi)
